# 🏠 House Price Prediction — Linear Regression

**Dataset:** [Home Value Insights — House Price Regression Dataset](https://www.kaggle.com/datasets/prokshitha/home-value-insights) by *prokshitha*

An end-to-end linear regression workflow:

1. Load the data (directly from Kaggle)
2. Explore it (EDA)
3. Preprocess & split into train / test
4. Train a **Linear Regression** model
5. **Evaluate** thoroughly — R², Adjusted R², MAE, MSE, RMSE, MAPE, cross-validation & residual analysis
6. Predict the price of a brand-new house

**Target variable:** `House_Price`

| Column | Meaning |
|---|---|
| `Square_Footage` | Interior size of the house (sq ft) |
| `Num_Bedrooms` | Number of bedrooms |
| `Num_Bathrooms` | Number of bathrooms |
| `Year_Built` | Year the house was built |
| `Lot_Size` | Size of the lot (acres) |
| `Garage_Size` | Garage capacity (number of cars) |
| `Neighborhood_Quality` | Neighborhood quality rating (1–10) |
| `House_Price` | **Sale price in USD (target)** |


## 1. Setup

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded.")

## 2. Load the data

The cell below pulls the dataset straight from Kaggle with `kagglehub`.
If that doesn't work (no internet / credentials), upload
`house_price_regression_dataset.csv` using the 📁 **Files** panel on the left —
the cell will then pick it up automatically.

In [ ]:
!pip install -q kagglehub

import os
import pandas as pd

df = None

# --- Option A: download directly from Kaggle via kagglehub ---
try:
    import kagglehub
    path = kagglehub.dataset_download("prokshitha/home-value-insights")
    csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
    df = pd.read_csv(os.path.join(path, csv_files[0]))
    print(f"Loaded from Kaggle: {csv_files[0]}  ->  shape {df.shape}")
except Exception as e:
    print("kagglehub download didn't work; looking for a local CSV instead.")
    print("Reason:", e)

# --- Option B: fall back to a manually uploaded / local CSV ---
if df is None:
    local_candidates = [f for f in os.listdir(".") if f.endswith(".csv")]
    if local_candidates:
        df = pd.read_csv(local_candidates[0])
        print(f"Loaded local file: {local_candidates[0]}  ->  shape {df.shape}")
    else:
        raise FileNotFoundError(
            "No CSV found. Upload 'house_price_regression_dataset.csv' via the Files "
            "panel, or run:  from google.colab import files; files.upload()"
        )

df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
df.describe()

### Define the target and features

We predict `House_Price` from every other column. Pulling the feature list
from the data keeps the notebook working even if a column is added or renamed.

In [ ]:
TARGET = "House_Price"
features = [c for c in df.columns if c != TARGET]

print("Target  :", TARGET)
print("Features:", features)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df[TARGET], kde=True, bins=40, color="steelblue")
plt.title("Distribution of House Price")
plt.xlabel("House Price (USD)")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True,
            cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix")
plt.show()

print("Correlation with House_Price (sorted):")
print(corr[TARGET].sort_values(ascending=False))

In [ ]:
# Scatter plot of each feature against the target
n = len(features)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].scatter(df[col], df[TARGET], alpha=0.4, s=15, color="teal")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(TARGET)
    axes[i].set_title(f"{col} vs {TARGET}")

for j in range(i + 1, len(axes)):   # hide unused subplots
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 4. Preprocess & split

- `X` = features, `y` = target
- 80% train / 20% test split (fixed `random_state` for reproducibility)
- Standardize features so coefficients are comparable; note that scaling does
  **not** change linear-regression predictions or R².

In [ ]:
X = df[features]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set    : {X_test.shape[0]} rows")

# Fit the scaler on the training data only, then apply to both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Train the Linear Regression model

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Model trained.\n")
print(f"Intercept: {model.intercept_:,.2f}")

coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_,
}).sort_values("Coefficient", key=abs, ascending=False)

print("\nCoefficients (standardized features — larger |value| = stronger effect):")
coef_df

In [ ]:
plt.figure(figsize=(8, 5))
colors = ["seagreen" if c > 0 else "indianred" for c in coef_df["Coefficient"]]
plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Feature Coefficients (standardized)")
plt.xlabel("Coefficient")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Evaluation

Predict on the held-out test set and measure performance.

In [ ]:
y_pred = model.predict(X_test_scaled)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

# Adjusted R² accounts for the number of predictors
n = X_test.shape[0]
p = X_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("Test-set performance")
print("-" * 32)
print(f"R2              : {r2:.4f}")
print(f"Adjusted R2     : {adj_r2:.4f}")
print(f"MAE             : {mae:,.2f}")
print(f"MSE             : {mse:,.2f}")
print(f"RMSE            : {rmse:,.2f}")
print(f"MAPE            : {mape * 100:.2f}%")

**How to read these:**
- **R² / Adjusted R²** — share of price variation explained (1.0 = perfect). Adjusted R² penalizes features that don't pull their weight.
- **MAE** — average dollar error, easy to interpret.
- **RMSE** — like MAE but punishes big misses harder; same units (USD).
- **MAPE** — average error as a percentage of the true price.

In [ ]:
# 5-fold cross-validation for a more stable estimate of R²
# (scaling is omitted here because it doesn't affect linear-regression R²)
cv_scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")

print("5-fold cross-validated R2 scores:", np.round(cv_scores, 4))
print(f"Mean CV R2: {cv_scores.mean():.4f}  (+/- {cv_scores.std():.4f})")

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.5, color="royalblue", s=25)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, "r--", linewidth=2, label="Perfect prediction")
plt.xlabel("Actual Price (USD)")
plt.ylabel("Predicted Price (USD)")
plt.title("Actual vs Predicted")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Residuals vs predicted — should scatter randomly around 0
axes[0].scatter(y_pred, residuals, alpha=0.5, color="darkorange", s=25)
axes[0].axhline(0, color="red", linestyle="--", linewidth=2)
axes[0].set_xlabel("Predicted Price (USD)")
axes[0].set_ylabel("Residual (Actual - Predicted)")
axes[0].set_title("Residuals vs Predicted")

# Residual distribution — should look roughly normal, centered on 0
sns.histplot(residuals, kde=True, bins=40, color="mediumpurple", ax=axes[1])
axes[1].axvline(0, color="red", linestyle="--", linewidth=2)
axes[1].set_xlabel("Residual (USD)")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()

## 7. Predict the price of a new house

Edit the values below to describe a house and get a predicted price.

In [ ]:
# Describe a new house (edit these values)
new_house = pd.DataFrame([{
    "Square_Footage": 2500,
    "Num_Bedrooms": 4,
    "Num_Bathrooms": 3,
    "Year_Built": 2005,
    "Lot_Size": 2.5,
    "Garage_Size": 2,
    "Neighborhood_Quality": 8,
}])

# Keep only the trained columns, in the right order, then scale with the SAME scaler
new_house = new_house[features]
new_house_scaled = scaler.transform(new_house)
predicted_price = model.predict(new_house_scaled)[0]

print("New house:")
print(new_house.T)
print(f"\nPredicted price: ${predicted_price:,.0f}")

## Summary

- Loaded the Home Value Insights dataset and explored it with EDA.
- Trained a **Linear Regression** model on an 80/20 split.
- Evaluated with R², Adjusted R², MAE, MSE, RMSE, MAPE, and 5-fold cross-validation.
- Checked residuals to validate the model's assumptions.

**Next steps to try:** add polynomial features, apply `Ridge` / `Lasso` regularization,
or compare against `RandomForestRegressor` or `GradientBoostingRegressor`.